# Loans and credit facilities

**Prerequisites:** **05**. Native **`term_loan`** and **`revolving_credit`** JSON cover draws, commitment/usage fees, and delayed-draw style availability windows where modeled.


## Concept

- **Term loan:** fixed or floating margin; amortization rules.
- **Revolver:** deterministic draw/repay paths; **commitment** and **usage** fees on drawn vs undrawn.
- **DDTL:** delayed-draw tranche inside `term_loan` encodes availability windows.

**IRR** (money-weighted return) is the rate that zeros PV of all cashflows—conceptually parallel to **discount margin** / **all-in yield** metrics when exposed.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (loans / RCF).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

term_loan = {
    "type": "term_loan",
    "spec": {
        "id": "TERM-LOAN-USD-5Y",
        "notional_limit": {"amount": "10000000", "currency": "USD"},
        "currency": "USD",
        "issue_date": "2024-01-01",
        "maturity": "2029-01-01",
        "discount_curve_id": "USD-OIS",
        "rate": {"Fixed": {"rate_bp": 600}},
        "day_count": "Act360",
        "frequency": {"count": 3, "unit": "months"},
        "bdc": "modified_following",
        "calendar_id": None,
        "stub": "None",
        "amortization": {"PercentPerPeriod": {"bp": 250}},
        "coupon_type": "Cash",
        "settlement_days": 2,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

revolver = {
    "type": "revolving_credit",
    "spec": {
        "id": "RCF-USD-3Y",
        "commitment_amount": {"amount": "50000000", "currency": "USD"},
        "drawn_amount": {"amount": "10000000", "currency": "USD"},
        "commitment_date": "2024-01-01",
        "maturity": "2027-01-01",
        "discount_curve_id": "USD-OIS",
        "day_count": "Act360",
        "frequency": {"count": 3, "unit": "months"},
        "stub": "ShortFront",
        "recovery_rate": 0.0,
        "base_rate_spec": {
            "Floating": {
                "index_id": "USD-SOFR-3M",
                "spread_bp": "250",
                "gearing": "1",
                "gearing_includes_spread": True,
                "floor_bp": "0",
                "all_in_floor_bp": None,
                "cap_bp": None,
                "index_cap_bp": None,
                "dc": "Act360",
                "bdc": "modified_following",
                "calendar_id": "weekends_only",
                "fixing_calendar_id": None,
                "reset_freq": {"count": 3, "unit": "months"},
                "reset_lag_days": 2,
                "payment_lag_days": 0,
                "end_of_month": False,
            }
        },
        "draw_repay_spec": {
            "Deterministic": [
                {"date": "2024-03-01", "amount": {"amount": "5000000", "currency": "USD"}, "is_draw": True},
                {"date": "2025-06-01", "amount": {"amount": "3000000", "currency": "USD"}, "is_draw": False},
            ]
        },
        "fees": {
            "commitment_fee_tiers": [{"threshold": "0", "bps": "25"}],
            "usage_fee_tiers": [{"threshold": "0", "bps": "10"}],
            "facility_fee_bp": 5.0,
        },
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

term_json = json.dumps(term_loan)
rev_json = json.dumps(revolver)
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    try:
        validate_instrument_json(js)
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)

print(
    "IRR concept: solve r with sum(CF_t / (1+r)^t)=0 on the facility cashflow schedule (fees + interest + draws/repays)."
)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("market ready")


## Pricing


In [ ]:
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    try:
        out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


## Metrics


In [ ]:
for label, js, mets in (
    ("term_loan", term_json, ["dv01", "effective_rate"]),
    ("revolving_credit", rev_json, ["dv01", "discount_margin", "effective_rate", "all_in_rate"]),
):
    try:
        out = price_instrument_with_metrics(
            js, market_json, AS_OF_STR, model="discounting", metrics=mets
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in mets:
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)


## Takeaways

- **Term loans** and **RCFs** share the same JSON pricing pipeline as simpler rates instruments.
- **Fee stacks** (commitment / usage / facility) interact with draws to shape IRR and PV.
- If a schema field moves between releases, **`validate_instrument_json`** is the first line of defense.
